# Physical Layer Guide

This notebook demonstrates the Physical Layer features: qubit creation, EPR pair creation, fidelity measurement, and the entanglement creation heralding protocol (ECHP).

## Imports and Setup

In [ ]:
from quantumnet.topology import Network, Host
from quantumnet.quantum import Qubit, Epr
from quantumnet.utils import Logger

Logger.get_instance().activate()

In [ ]:
network = Network()
network.set_ready_topology("Grade", 3, 3)
network.draw()

## 1. Creating Qubits

`physical.create_qubit(host_id)` creates a qubit and adds it to the specified host's memory. Each qubit is assigned a unique ID and a random initial fidelity.

In [ ]:
host0 = network.get_host(0)
print(f"Memory before: {len(host0.memory)} qubits")

network.physical.create_qubit(0)

print(f"Memory after: {len(host0.memory)} qubits")
last_qubit = host0.memory[-1]
print(f"Last qubit: id={last_qubit.qubit_id}, fidelity={last_qubit.get_current_fidelity():.4f}")

## 2. Creating EPR Pairs

`physical.create_epr_pair(fidelity)` creates an EPR pair with the given initial fidelity (default 1.0).

In [ ]:
epr = network.physical.create_epr_pair(fidelity=0.95)
print(f"EPR ID: {epr.epr_id}")
print(f"Initial fidelity: {epr.get_initial_fidelity():.4f}")
print(f"Current fidelity: {epr.get_current_fidelity():.4f}")

## 3. Adding / Removing EPRs from Channels

EPR pairs can be added to or removed from a specific network channel (edge).

In [ ]:
channel = (0, 1)
print(f"EPRs on channel {channel} before: {len(network.get_eprs_from_edge(*channel))}")

# Add an EPR to the channel
new_epr = network.physical.create_epr_pair(fidelity=0.9)
network.physical.add_epr_to_channel(new_epr, channel)
print(f"EPRs on channel {channel} after add: {len(network.get_eprs_from_edge(*channel))}")

# Remove it
network.physical.remove_epr_from_channel(new_epr, channel)
print(f"EPRs on channel {channel} after remove: {len(network.get_eprs_from_edge(*channel))}")

## 4. Fidelity Measurement

`fidelity_measurement_only_one(qubit)` measures (and applies decoherence to) a single qubit.  
`fidelity_measurement(q1, q2)` measures both and returns their combined fidelity.

In [ ]:
q = host0.memory[0]
print(f"Qubit fidelity before measurement: {q.get_current_fidelity():.6f}")

measured = network.physical.fidelity_measurement_only_one(q)
print(f"Qubit fidelity after measurement: {measured:.6f}")

## 5. Entanglement Creation Heralding Protocol (ECHP)

The ECHP consumes one qubit from each host and attempts to create an EPR pair. Returns `True` if the resulting fidelity meets the configured threshold.

In [ ]:
alice = network.get_host(3)
bob = network.get_host(4)

result = network.physical.entanglement_creation_heralding_protocol(alice, bob)
print(f"ECHP result: {result}")
print(f"Current timeslot: {network.clock.now}")

## 6. On-Demand and Replay ECHP

`echp_on_demand` and `echp_on_replay` use channel-specific success probabilities.

In [ ]:
result_demand = network.physical.echp_on_demand(3, 4)
print(f"On-demand ECHP result: {result_demand}")

result_replay = network.physical.echp_on_replay(3, 4)
print(f"Replay ECHP result: {result_replay}")

## 7. Resource Counters

In [ ]:
print(f"Used qubits (physical layer): {network.physical.get_used_qubits()}")
print(f"Used EPRs (physical layer): {network.physical.get_used_eprs()}")
print(f"Current timeslot: {network.clock.now}")